In [10]:
import numpy as np
import pandas as pd
import balancepy as bp
import balancepy.anaropia as bpa
import balancepy.anaropia_project as bpap
from pathlib import Path
import sys; sys.path.insert(0, '..')
from src.config import paths

# Display dataset information from metadata file and quality checks

In [11]:
# run syncronization with quality flag file
# bp.sync_quality_flags_from_metadata(paths=paths)

metadata = bpap.get_metadata(paths=paths)
condition_cols = [
    col for col in metadata.columns
    if isinstance(col, str) and col.startswith('c')
]
metadata[condition_cols] = metadata[condition_cols].fillna(-1)


# Load and preprocess data across all subjects

In [12]:
# preprocessing configuration
preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 210
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 180
preproc_config.cycle_start_samples = 878
print(preproc_config)

# stimulus-response class configuration
sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch', 
    response_name=bpa.COM_LEGACY_AP, 
    frequency_selection='all'
    )
print(sr_config)

AnaropiaPreprocessingConfig(
  samplingrate_Hz: 90
  resample: True
  end_time_seconds: 210
  filter_type: None
  filter_order: 2
  filter_cutoff_Hz: 8
  cut_to_cycles: True
  cycle_start_samples: 878
  cycle_length_samples: 180
)
AnaropiaSRDataConfig(stimulus_name='stim_pitch', response_name=COMConfig(shoulder_pos_column='shld_zpos', hip_pos_column='hip_zpos', shoulder_height_column='shld_ypos', hip_height_column='hip_ypos', rotation=True), frequency_selection='all', name=None, column_scales=None)


In [13]:
data_all_conditions = []
condition_groups = [
    ([1, 13], None),
    ([15, 16, 17], 0.03),
    ([4, 5], 0.05),
    (6, 0.15),
    (7, 0.25),
    (8, 0.5),
    (9, 1),
    (10, 5),
    (11, 10),
    ([2, 3, 14], 0.01),
    ([18, 19], 0.07),
    (20, 0.1),
    (21, 0.2),
    (22, 0.3),
]

for condition, velocity in condition_groups:
    sr_data_list = []

    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths=paths,
        conditions=condition,
        output_columns=['body_height_m'],
    ):
        print(f"Loading {filepath} for subject {subject_id} and condition {condition}...")
        sr = bpa.build_sr_data(
            filename=filepath,
            body_height_m=body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=f"{subject_id}_{condition}",
        )
        sr_data_list.append(sr)
    merged = bp.merge_sr_data(sr_data_list)
    merged.name = sr_data_list[0].name[13:]
    merged.velocity = velocity
    data_all_conditions.append(merged)




In [22]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ordered_conditions = sorted(
    data_all_conditions,
    key=lambda group: (
        group.velocity is not None,
        float('inf') if group.velocity is None else group.velocity,
    ),
)

fig = make_subplots(
    rows=2,
    cols=len(ordered_conditions),
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.01,
    vertical_spacing=0.08,
    subplot_titles=[
        'No stim' if group.velocity is None else str(group.velocity)
        for group in ordered_conditions
    ],
)

for col, group in enumerate(ordered_conditions, start=1):
    time = group.time
    stim = group.stimulus_mean
    resp = group.response_mean0

    fig.add_trace(
        go.Scatter(x=time, y=stim, mode='lines', showlegend=False),
        row=1, col=col
    )
    fig.add_trace(
        go.Scatter(x=time, y=resp, mode='lines', showlegend=False),
        row=2, col=col
    )

for col in range(1, len(ordered_conditions) + 1):
    fig.update_xaxes(title_text='time (s)', row=2, col=col)
    fig.update_yaxes(range=[-0.05, 0.05], row=2, col=col)

fig.update_yaxes(title_text='stimulus', row=1, col=1)
fig.update_yaxes(title_text='response', row=2, col=1)

fig.update_layout(
    height=900,
    width=max(2400, 220 * len(ordered_conditions)),
    title='Condition-Wise Mean Stimulus (row 1) and Response (row 2)',
    template='plotly_white',
)

fig.show()

In [20]:
gain_conditions = sorted(
    [group for group in data_all_conditions if group.velocity is not None],
    key=lambda group: group.velocity,
)
velocity = np.array([group.velocity for group in gain_conditions])

gain = [group.gain[0] for group in gain_conditions]

fig = go.Figure()
fig.add_trace(go.Scatter(x=velocity, y=gain, mode="markers", showlegend=False))
fig.update_xaxes(type="log", title_text="velocity")
fig.update_yaxes(type="log", title_text="gain")
fig.show()

In [21]:
resp_conditions = [group for group in data_all_conditions if group.velocity is not None]
velocity = np.array([group.velocity for group in resp_conditions])

resp_std = [np.abs(group.response_spectrum_mean[0]) for group in resp_conditions]

fig = go.Figure()
fig.add_trace(go.Scatter(x=velocity, y=resp_std, mode="markers", showlegend=False))
fig.update_xaxes(type="log", title_text="velocity")
fig.update_yaxes(type="log", title_text="response_std")
fig.show()